# Alzheimer's MRI Classification — Three-Model Comparison

This notebook trains and compares the three architectures supplied in the original notebook:

1. **Base Model — Kaustubhi**
2. **CNN Model — Samiksha**
3. **ResNet50**

**Architecture policy:** The model layers, filter sizes, dense layers, optimizers, learning rates, dropout values, and other architecture/training settings from the supplied file are retained. The notebook is reorganized for consistent execution, evaluation, plots, and comparison.

## Dataset layout

```text
Data/
└── Alzheimerr/
    ├── train_set/
    │   ├── MildDemented/
    │   ├── ModerateDemented/
    │   ├── NonDemented/
    │   └── VeryMildDemented/
    └── test_set/
        └── (same four class folders)
```

Update `DATA_ROOT` below if your dataset is stored elsewhere.

## 1. Imports and reproducibility

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 2. Paths and output folders

In [ ]:
DATA_ROOT = "../Data/Alzheimerr"
TRAIN_DIR = os.path.join(DATA_ROOT, "train_set")
TEST_DIR = os.path.join(DATA_ROOT, "test_set")

OUTPUT_DIR = "../Outputs/Model_Comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(TRAIN_DIR), f"Training folder not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Test folder not found: {TEST_DIR}"

print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("Output directory:", OUTPUT_DIR)

## 3. Data generators — Kaustubhi and Samiksha

Both 128 × 128 models use the same generator configuration from the supplied notebook:
- Rescaling by `1./255`
- Rotation, width/height shift, and zoom augmentation for training
- 20% validation split
- Batch size 32

In [ ]:
train_datagen_128 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    validation_split=0.2
)

validation_datagen_128 = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

test_datagen_128 = ImageDataGenerator(
    rescale=1./255
)

train_gen_128 = train_datagen_128.flow_from_directory(
    TRAIN_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    seed=SEED
)

val_gen_128 = validation_datagen_128.flow_from_directory(
    TRAIN_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED
)

test_gen_128 = test_datagen_128.flow_from_directory(
    TEST_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_gen_128.class_indices.keys())

print("Class mapping:", train_gen_128.class_indices)
print("Class names:", class_names)
assert train_gen_128.class_indices == val_gen_128.class_indices == test_gen_128.class_indices

## 4. Shared evaluation and plotting functions
These functions do not alter any model architecture. They store the training history and test metrics for the final comparison.

In [ ]:
results = {}
histories = {}
trained_models = {}

def plot_training_history(history, model_name):
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history["loss"], label="Training Loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{safe_name}_training_curves.png"), dpi=150)
    plt.show()


def evaluate_and_store(model, test_generator, model_name, history):
    test_generator.reset()
    test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

    test_generator.reset()
    probabilities = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)
    y_true = test_generator.classes

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    results[model_name] = {
        "Test Accuracy": float(test_accuracy),
        "Test Loss": float(test_loss),
        "Precision (macro)": report_dict["macro avg"]["precision"],
        "Recall (macro)": report_dict["macro avg"]["recall"],
        "F1-score (macro)": report_dict["macro avg"]["f1-score"]
    }

    histories[model_name] = history
    trained_models[model_name] = model

    print(f"\n{model_name} — Classification Report")
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(7, 6))
    display.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
    ax.set_title(f"{model_name} - Test Confusion Matrix")
    plt.tight_layout()

    safe_name = model_name.lower().replace(" ", "_")
    plt.savefig(os.path.join(OUTPUT_DIR, f"{safe_name}_confusion_matrix.png"), dpi=150)
    plt.show()

    plot_training_history(history, model_name)

    return test_loss, test_accuracy

# Model 1 — Base Model Kaustubhi

The architecture below is retained exactly from the supplied notebook:
- Convolution filters: 16 → 32 → 64 → 128
- Batch Normalization and ReLU in each block
- Max Pooling after the first three blocks
- Global Average Pooling
- Dropout 0.4
- Four-class Softmax output
- AdamW, learning rate 0.001, weight decay 0.0001
- Early stopping patience 2

In [ ]:
kaustubhi_model = models.Sequential([

    layers.Input(shape=(128, 128, 3)),

    # Block 1
    layers.Conv2D(16, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 4
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.GlobalAveragePooling2D(),

    # Classifier
    layers.Dropout(0.4),
    layers.Dense(4, activation='softmax')
])

kaustubhi_model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

kaustubhi_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
        verbose=1
    )
]

kaustubhi_model.summary()

In [ ]:
kaustubhi_history = kaustubhi_model.fit(
    train_gen_128,
    validation_data=val_gen_128,
    epochs=30,
    callbacks=kaustubhi_callbacks
)

In [ ]:
kaustubhi_model.save(os.path.join(OUTPUT_DIR, "kaustubhi_base_model.keras"))

evaluate_and_store(
    kaustubhi_model,
    test_gen_128,
    "Kaustubhi Base Model",
    kaustubhi_history
)

# Model 2 — CNN Model Samiksha

The architecture below is retained exactly from the supplied notebook:
- Two convolution layers in each of three blocks
- Filters: 32 → 64 → 128
- ReLU activation in every convolution
- Max Pooling after each block
- Flatten
- Four-class Softmax output
- Adam, learning rate 0.001
- Early stopping patience 6

In [ ]:
samiksha_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Block 1
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(4, activation='softmax')
])

samiksha_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

samiksha_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
]

samiksha_model.summary()

In [ ]:
samiksha_history = samiksha_model.fit(
    train_gen_128,
    validation_data=val_gen_128,
    epochs=40,
    callbacks=samiksha_callbacks
)

In [ ]:
samiksha_model.save(os.path.join(OUTPUT_DIR, "samiksha_cnn_model.keras"))

evaluate_and_store(
    samiksha_model,
    test_gen_128,
    "Samiksha CNN",
    samiksha_history
)

# Model 3 — ResNet50

The ResNet50 architecture from the supplied notebook is retained:
- ImageNet pretrained ResNet50 backbone
- `include_top=False`
- Backbone initially frozen
- Global Average Pooling
- Batch Normalization
- Dropout 0.4
- Dense 256 with ReLU
- Dropout 0.3
- Four-class Softmax output
- Adam, learning rate 0.001
- Early stopping patience 6

**Preprocessing note:** ResNet50 uses its own `preprocess_input`; it must not use the `rescale=1./255` preprocessing used by the two 128 × 128 CNNs. ResNet50 input size is 224 × 224, as in the supplied notebook.

In [ ]:
train_datagen_resnet = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    validation_split=0.2
)

validation_datagen_resnet = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    validation_split=0.2
)

test_datagen_resnet = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)

train_gen_resnet = train_datagen_resnet.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen_resnet = validation_datagen_resnet.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

test_gen_resnet = test_datagen_resnet.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

assert train_gen_resnet.class_indices == val_gen_resnet.class_indices == test_gen_resnet.class_indices
print("ResNet50 class mapping:", train_gen_resnet.class_indices)

In [ ]:
resnet_base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

resnet_base_model.trainable = False

resnet50_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    resnet_base_model,

    layers.GlobalAveragePooling2D(),

    layers.BatchNormalization(),

    layers.Dropout(0.4),

    layers.Dense(256, activation='relu'),

    layers.Dropout(0.3),

    layers.Dense(4, activation='softmax')
])

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet50_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
]

resnet50_model.summary()

In [ ]:
resnet50_history = resnet50_model.fit(
    train_gen_resnet,
    validation_data=val_gen_resnet,
    epochs=40,
    callbacks=resnet50_callbacks
)

In [ ]:
resnet50_model.save(os.path.join(OUTPUT_DIR, "resnet50_model.keras"))

evaluate_and_store(
    resnet50_model,
    test_gen_resnet,
    "ResNet50",
    resnet50_history
)

# 7. Final comparison of all three models

The comparison uses the same held-out test set and reports:
- Test accuracy
- Test loss
- Macro precision
- Macro recall
- Macro F1-score

The two CNNs use the 128 × 128 generator, while ResNet50 uses the same test directory resized to 224 × 224 with ResNet50 preprocessing.

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values("Test Accuracy", ascending=False)

display(comparison_df.round(4))

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "three_model_comparison.csv"),
    index=True
)

In [ ]:
# Compare test accuracy
plt.figure(figsize=(9, 5))
plt.bar(comparison_df.index, comparison_df["Test Accuracy"])
plt.title("Test Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_test_accuracy_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Compare test loss
plt.figure(figsize=(9, 5))
plt.bar(comparison_df.index, comparison_df["Test Loss"])
plt.title("Test Loss Comparison")
plt.xlabel("Model")
plt.ylabel("Test Loss")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_test_loss_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Compare macro precision, recall, and F1-score
comparison_df[[
    "Precision (macro)",
    "Recall (macro)",
    "F1-score (macro)"
]].plot(kind="bar", figsize=(10, 5))

plt.title("Macro Precision, Recall and F1-score Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_precision_recall_f1_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Compare validation accuracy curves
plt.figure(figsize=(10, 6))

for model_name, history in histories.items():
    plt.plot(
        range(1, len(history.history["val_accuracy"]) + 1),
        history.history["val_accuracy"],
        label=model_name
    )

plt.title("Validation Accuracy Across Models")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "validation_accuracy_all_models.png"), dpi=150)
plt.show()

## 8. Saved outputs

After all cells run successfully, the `../Outputs/Model_Comparison/` folder contains:
- Three trained `.keras` model files
- Training accuracy/loss plots for each model
- Test confusion matrix plots for each model
- Test comparison CSV
- Test accuracy, test loss, precision/recall/F1 comparison plots
- Combined validation accuracy plot

## Notes for interpreting the comparison

1. Compare models using the **same test population** and the same class mapping.
2. The input resolution and preprocessing differ by design because those are part of the supplied model pipelines.
3. The architectures and training settings above have been retained from your uploaded notebook. No ResNet fine-tuning stage has been added.
4. For medical-image classification, inspect per-class recall and confusion matrices in addition to overall accuracy.
5. If multiple images from one patient can appear in different splits, use a patient-level split before drawing conclusions; otherwise test results may be optimistic.